In [205]:
import pandas as pd
import numpy as np
import functions as fn
import PairwiseComparison as pc

np.set_printoptions(edgeitems=10, infstr='inf',
linewidth=300, nanstr='nan', precision=8,
suppress=False, threshold=1000, formatter=None)

In [206]:
Data_2014 = pd.read_csv('DataFiles/2014_game_results.csv')
Data_2015 = pd.read_csv('DataFiles/2015_game_results.csv')
Data_2016 = pd.read_csv('DataFiles/2016_game_results.csv')
Data_2017 = pd.read_csv('DataFiles/2017_game_results.csv')
Data_2018 = pd.read_csv('DataFiles/2018_game_results.csv')
Data_2019 = pd.read_csv('DataFiles/2019_game_results.csv')
Data_2021 = pd.read_csv('DataFiles/2021_game_results.csv')
Data_2022 = pd.read_csv('DataFiles/2022_game_results.csv')
Data_2023 = pd.read_csv('DataFiles/2023_game_results.csv')
Data_2024 = pd.read_csv('DataFiles/2024_game_results.csv')
Data_2025 = pd.read_csv('DataFiles/2025_game_results.csv')

Years = [2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025]
Data_Collection = [Data_2014, Data_2015, Data_2016, Data_2017, Data_2018, Data_2019, Data_2021, Data_2022, Data_2023, Data_2024, Data_2025]
season_length = [14,13,14,14,14,15,14,14,14,15,15]


# Set up/Merge Datasets
cleaned_seasons = []
for year, dataset, l in zip(Years, Data_Collection, season_length):
    cleaned = fn.clean_season(dataset, year)
    cleaned = cleaned[cleaned['Wk']<=l]
    cleaned_seasons.append(cleaned)

all_games = pd.concat(cleaned_seasons, ignore_index=True)
all_games = all_games.sort_values('Date', kind='stable').reset_index(drop=True)

SEC1=["Alabama","Arkansas","Auburn","Florida","Georgia","Kentucky","Louisiana State","Mississippi State","Mississippi","Missouri","South Carolina","Tennessee","Texas A&M","Vanderbilt"]
SEC2=["Alabama","Arkansas","Auburn","Florida","Georgia","Kentucky","Louisiana State","Mississippi State","Mississippi","Missouri","Oklahoma", "South Carolina","Tennessee","Texas","Texas A&M","Vanderbilt"]
BIG1=["Illinois","Indiana","Iowa","Maryland","Michigan","Michigan State","Minnesota", "Nebraska","Northwestern","Ohio State","Penn State","Purdue","Rutgers","Wisconsin"]
BIG2=["Illinois","Indiana","Iowa","Maryland","Michigan","Michigan State","Minnesota", "Nebraska","Northwestern","Ohio State","Oregon","Penn State","Purdue","Rutgers","Southern California","UCLA","Washington","Wisconsin"]

Teams1 = SEC1+BIG1
Teams2 = SEC2+BIG2

In [207]:
reduced_seasons=[]
for year, dataset in zip(Years, cleaned_seasons):
    if year<=2023:
        cleaned = dataset[dataset.Winner.isin(Teams1)]
        cleaned = cleaned[cleaned.Loser.isin(Teams1)]
    else:
        cleaned = dataset[dataset.Winner.isin(Teams2)]
        cleaned = cleaned[cleaned.Loser.isin(Teams2)]

    reduced_seasons.append(cleaned)

all_games = pd.concat(reduced_seasons, ignore_index=True)
all_games = all_games.sort_values('Date', kind='stable').reset_index(drop=True)

In [208]:
score_matrices=[]
for year, dataset in zip(Years, reduced_seasons):
    if year <=2023:
        Teams = Teams1
    else:
        Teams=Teams2
    W=np.zeros((len(Teams),len(Teams)))
    for i in range(len(Teams)):
        teamdf = dataset[dataset.Winner == Teams[i]]
        for j in range(len(Teams)):
            game = teamdf[teamdf.Loser == Teams[j]]
            if not game.empty:
                W[i,j]=sum(game.PtsW)
                W[j,i]=sum(game.PtsL)
    score_matrices.append(W)


In [209]:
exp = pc.RankZermelo(score_matrices[9],iterlimit = 1000,powers=True)
exr = np.argsort(np.flip(np.argsort(exp)))
example = pd.DataFrame({"Team":Teams2, "power":exp, "rank":exr})
example = example.sort_values(by = "rank")
print(example)

                   Team     power  rank
25           Ohio State  1.000000     0
13                Texas  0.947852     1
26               Oregon  0.818642     2
0               Alabama  0.791763     3
8           Mississippi  0.786154     4
11       South Carolina  0.780580     5
4               Georgia  0.723639     6
27           Penn State  0.701980     7
17              Indiana  0.681366     8
12            Tennessee  0.646323     9
14            Texas A&M  0.606932    10
6       Louisiana State  0.580983    11
3               Florida  0.557984    12
15           Vanderbilt  0.539764    13
1              Arkansas  0.471018    14
18                 Iowa  0.467528    15
9              Missouri  0.441473    16
30  Southern California  0.436717    17
20             Michigan  0.419030    18
22            Minnesota  0.416410    19
2                Auburn  0.406923    20
10             Oklahoma  0.386534    21
32           Washington  0.360601    22
5              Kentucky  0.344231    23


In [210]:
results = pd.DataFrame(columns=("ICG","difp","difr","dif3p","dif3r"))
for year, scores in zip(Years,score_matrices):
    if year <=2023:
        Teams = Teams1
        n=14
        m=14
    else:
        Teams=Teams2
        n=16
        m=18
    icgScores1 = scores[n:,:n]
    icgScores2 = scores[:n,n:]
    ICG = max(np.count_nonzero(icgScores1),np.count_nonzero(icgScores2))
    p = pc.RankZermelo(scores,iterlimit=1000,powers=True)
    r = np.argsort(np.flip(np.argsort(p)))
    difr=pc.rankDistance(r,n,m)
    difp=pc.scoreDistance(p,n,m)
    dif3r=pc.rankDistance(r,n,m,3)
    dif3p=pc.scoreDistance(p,n,m,3)
    res = pd.DataFrame({"ICG":ICG,"difp":difp,"difr":difr,"dif3p":dif3p,"dif3r":dif3r}, index=[year])
    results = pd.concat((results,res))
print(results)

C:\Users\jonmc\AppData\Local\Temp\ipykernel_47728\1281400031.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat((results,res))


     ICG      difp      difr     dif3p     dif3r
2014   2 -0.005968 -0.857143 -0.053344  1.000000
2015   1  0.088835 -3.142857  0.155038 -2.333333
2016   1  0.006639 -2.571429 -0.152128  1.666667
2017   2 -0.281890  9.428571 -0.459575  6.666667
2018   1 -0.002204  1.428571  0.078967  0.000000
2019   1  0.029153 -2.142857 -0.012223 -1.666667
2021   1  0.000111  2.285714  0.066217  2.666667
2022   1 -0.148237  6.000000 -0.286568  3.666667
2023   0 -0.137048  3.571429 -0.520594  3.666667
2024   4  0.165852 -8.500000  0.001716 -0.333333
2025   3  0.023982 -6.138889 -0.324896  3.000000


In [212]:
#d0=pc.difDistribution(14,14,0,t=-1,reps=5000,bias = 0,comp=True)
#d1=pc.difDistribution(14,14,1,t=-1,reps=5000,bias = 0,comp=True)
#d2=pc.difDistribution(14,14,2,t=-1,reps=5000,bias = 0,comp=True)
#d3=pc.difDistribution(16,18,3,t=-1,reps=5000,bias = 0,comp=True)
#d4=pc.difDistribution(16,18,4,t=-1,reps=5000,bias = 0,comp=True)

d0 = np.genfromtxt("DataFiles/d0.csv",delimiter = ',')
d1 = np.genfromtxt("DataFiles/d1.csv",delimiter = ',')
d2 = np.genfromtxt("DataFiles/d2.csv",delimiter = ',')
d3 = np.genfromtxt("DataFiles/d3.csv",delimiter = ',')
d4 = np.genfromtxt("DataFiles/d4.csv",delimiter = ',')

dlist = [d0,d1,d2,d3,d4]



In [213]:
#d30=pc.difDistribution(14,14,0,t=3,reps=5000,bias = 0,comp=True)
#d31=pc.difDistribution(14,14,1,t=3,reps=5000,bias = 0,comp=True)
#d32=pc.difDistribution(14,14,2,t=3,reps=5000,bias = 0,comp=True)
#d33=pc.difDistribution(16,18,3,t=3,reps=5000,bias = 0,comp=True)
#d34=pc.difDistribution(16,18,4,t=3,reps=5000,bias = 0,comp=True)

d30 = np.genfromtxt("DataFiles/d30.csv",delimiter = ',')
d31 = np.genfromtxt("DataFiles/d31.csv",delimiter = ',')
d32 = np.genfromtxt("DataFiles/d32.csv",delimiter = ',')
d33 = np.genfromtxt("DataFiles/d33.csv",delimiter = ',')
d34 = np.genfromtxt("DataFiles/d34.csv",delimiter = ',')

d3list = [d30,d31,d32,d33,d34]

In [214]:
#dr0=pc.difRankDistribution(14,14,0,t=-1,reps=5000,bias = 0,comp=True)
#dr1=pc.difRankDistribution(14,14,1,t=-1,reps=5000,bias = 0,comp=True)
#dr2=pc.difRankDistribution(14,14,2,t=-1,reps=5000,bias = 0,comp=True)
#dr3=pc.difRankDistribution(16,18,3,t=-1,reps=5000,bias = 0,comp=True)
#dr4=pc.difRankDistribution(16,18,4,t=-1,reps=5000,bias = 0,comp=True)

dr0 = np.genfromtxt("DataFiles/dr0.csv",delimiter = ',')
dr1 = np.genfromtxt("DataFiles/dr1.csv",delimiter = ',')
dr2 = np.genfromtxt("DataFiles/dr2.csv",delimiter = ',')
dr3 = np.genfromtxt("DataFiles/dr3.csv",delimiter = ',')
dr4 = np.genfromtxt("DataFiles/dr4.csv",delimiter = ',')

drlist = [dr0,dr1,dr2,dr3,dr4]

In [ ]:
dfp = results["difp"]
dfICG = results["ICG"]
for i in Years:
    ICG = dfICG.get(i)
    difp = dfp.get(i)
    d=dlist[ICG]
    [x,y]=pc.get_ecdf(d)
    idx =np.argmax(x>difp)
    p=y[idx]
    print(i)
    print(difp)
    print(1-p)

2014
-0.005968943927372572
0.543508701740348
2015
0.08883539702179805
0.2608521704340868
2016
0.006637186358882774
0.49149829965993197
2017
-0.28189020972284007
0.9967993598719744
2018
-0.00220486191252528
0.5127025405081016
2019
0.029152060462678486
0.4282856571314263
2021
0.00011054498859947026
0.5061012202440488
2022
-0.14823760708214795
0.8651730346069214
2023
-0.06317778501384452
0.6019203840768154
2024
0.16585172566601952
0.026005201040208092
2025
0.02398252716574889
0.3866773354670934


In [ ]:

dfr = results["difr"]
dfICG = results["ICG"]
for i in Years:    
    ICG = dfICG.get(i)
    difr = dfr.get(i)
    d=drlist[ICG]
    [x,y]=pc.get_ecdf(d)
    idx =np.argmax(x>difr)
    p=y[idx]
    print(i)
    print(difr)
    print(p)

2014
-0.8571428571428577
0.24084816963392677
2015
-3.1428571428571423
0.11682336467293458
2016
-2.571428571428573
0.15523104620924186
2017
9.428571428571429
0.9933986797359472
2018
1.428571428571427
0.741748349669934
2019
-2.1428571428571423
0.19383876775355072
2021
2.2857142857142847
0.8057611522304461
2022
6.0
0.9533906781356272
2023
-1.7142857142857153
0.28845769153830764
2024
-8.5
0.000600120024004801
2025
-6.138888888888889
0.007001400280056011


In [ ]:
dfp = results["dif3p"]
dfICG = results["ICG"]
for i in Years:
    ICG = dfICG.get(i)
    difp = dfp.get(i)
    d=d3list[ICG]
    [x,y]=pc.get_ecdf(d)
    idx =np.argmax(x>difp)
    p=y[idx]
    print(i)
    print(difp)
    print(1-p)

2014
-0.05334490979664264
0.6417283456691338
2015
0.1550383075825723
0.2518503700740148
2016
-0.15213041462987342
0.7479495899179835
2017
-0.45957530462217394
0.9975995199039808
2018
0.07896626594236489
0.3642728545709142
2019
-0.012224153129824389
0.5179035807161432
2021
0.06621702263017848
0.381876375275055
2022
-0.2865692984334154
0.8999799959991999
2023
-0.4039829040825528
0.8859771954390878
2024
0.0017154421492783234
0.504500900180036
2025
-0.32489546998224306
0.993998799759952


In [215]:
for i in range(5):
    np.savetxt("DataFiles/testd{}.csv".format(i), dlist[i], delimiter=",")
for i in range(5):
    np.savetxt("DataFiles/testd3{}.csv".format(i), d3list[i], delimiter=",")
for i in range(5):
    np.savetxt("DataFiles/testdr{}.csv".format(i), drlist[i], delimiter=",")
